# Task 01: LCEL Classification Chain

Build a ticket classification chain using LCEL. The chain should take a ticket text and return `{"category": ..., "priority": ...}`.

## Setup

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

OPENAI_API_KEY = "your-api-key-here"
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)
print("✓ LLM initialized")


In [ ]:
import json, os

# In Jupyter, os.getcwd() returns the notebook's directory (learning/, tasks/, solutions/)
# Fixtures are one level up at ../fixtures/input/
fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))

with open(os.path.join(fixtures, "tickets.json")) as f:
    tickets = json.load(f)

with open(os.path.join(fixtures, "knowledge_base.json")) as f:
    knowledge_base = json.load(f)

with open(os.path.join(fixtures, "test_questions.json")) as f:
    test_questions = json.load(f)

print(f"✓ Loaded {len(tickets)} tickets, {len(knowledge_base)} KB articles, {len(test_questions)} test questions")


## Part 1: Build the Classification Chain

Create a `chain` using LCEL that:
- Takes `{"ticket": <text>}` as input
- Returns a dict with `category` (billing/technical/account/shipping) and `priority` (high/medium/low)

Hint: add **priority definitions** to your system prompt — the model's default sense of urgency differs from the dataset labels.

In [ ]:
# YOUR CODE HERE
# Define: prompt, chain
# chain should accept {"ticket": str} and return dict



In [ ]:
# TEST — Structural check (no API call)
assert 'chain' in dir() or 'chain' in locals(), "Define a variable named 'chain'"
assert hasattr(chain, 'invoke'), "chain must have .invoke() method"
print("✓ chain has .invoke()")


## Part 2: Batch Classification

Implement `classify_all(chain, tickets) -> list[dict]` that classifies all 20 tickets using `.batch()`.

In [ ]:
# YOUR CODE HERE
def classify_all(chain, tickets):
    pass



In [ ]:
# TEST — Run classification on all 20 tickets
results = classify_all(chain, tickets)

assert isinstance(results, list), "classify_all must return a list"
assert len(results) == len(tickets), f"Expected {len(tickets)} results, got {len(results)}"

VALID_CATEGORIES = {"billing", "technical", "account", "shipping"}
VALID_PRIORITIES = {"high", "medium", "low"}

for i, r in enumerate(results):
    assert isinstance(r, dict), f"Result {i} must be a dict"
    assert "category" in r, f"Result {i} missing 'category'"
    assert "priority" in r, f"Result {i} missing 'priority'"
    assert r["category"] in VALID_CATEGORIES, f"Result {i}: invalid category {r['category']!r}"
    assert r["priority"] in VALID_PRIORITIES, f"Result {i}: invalid priority {r['priority']!r}"

print(f"✓ All {len(results)} results have valid format")


In [ ]:
# TEST — Accuracy check (real LLM output)
correct_cat = sum(
    1 for r, t in zip(results, tickets)
    if r.get("category") == t["category"]
)
correct_pri = sum(
    1 for r, t in zip(results, tickets)
    if r.get("priority") == t["priority"]
)

cat_acc = correct_cat / len(tickets)
pri_acc = correct_pri / len(tickets)

print(f"Category accuracy: {cat_acc:.0%} ({correct_cat}/{len(tickets)})")
print(f"Priority accuracy:  {pri_acc:.0%} ({correct_pri}/{len(tickets)})")

# Show mistakes
for r, t in zip(results, tickets):
    if r.get("category") != t["category"] or r.get("priority") != t["priority"]:
        cat_mark = "✓" if r.get("category") == t["category"] else "✗"
        pri_mark = "✓" if r.get("priority") == t["priority"] else "✗"
        print(f"  ID {t['id']}: cat={cat_mark}{r.get('category')} pri={pri_mark}{r.get('priority')}  (expected: {t['category']}/{t['priority']})")

assert cat_acc >= 0.80, f"Category accuracy {cat_acc:.0%} < 80% — improve your prompt"
assert pri_acc >= 0.60, f"Priority accuracy {pri_acc:.0%} < 60% — add priority definitions to your prompt"
print("\n✓ Accuracy targets met!")


## Part 3: Chain-of-Thought Variant

Create `cot_chain` that includes a `reasoning` field explaining the classification before giving the final answer. This often improves accuracy.

In [ ]:
# YOUR CODE HERE
# Define cot_chain — same interface as chain but returns dict with "reasoning", "category", "priority"



In [ ]:
# TEST — CoT format check
sample_result = cot_chain.invoke({"ticket": "I was charged twice this billing cycle."})

assert isinstance(sample_result, dict), "cot_chain must return a dict"
assert "category" in sample_result, "Missing 'category'"
assert "priority" in sample_result, "Missing 'priority'"
assert "reasoning" in sample_result, "Missing 'reasoning' field — add chain-of-thought"
assert len(sample_result["reasoning"]) > 20, "'reasoning' should be a real explanation"

print("✓ CoT chain format correct")
print(f"  Reasoning: {sample_result['reasoning'][:80]}...")
print(f"  Category: {sample_result['category']}, Priority: {sample_result['priority']}")
